# QC Validation: Non-Cas9 CRISPR Data for inDecay

This notebook performs quality control and distribution sanity checks
on the preprocessed TREX2 and Cas12a datasets.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from pathlib import Path

from scripts.preprocess.common.indel_format import parse_indel_string, validate_indel_string
from scripts.preprocess.common.validation import validate_output_pair
from scripts.preprocess.config import TREX2_OUTPUT_DIR, CAS12A_OUTPUT_DIR, INDEL_PROFILES_CSV, REFERENCE_FASTA

## 1. Load Processed Data

In [ ]:
def load_profiles(output_dir):
    """Load indel profiles CSV."""
    csv_path = Path(output_dir) / INDEL_PROFILES_CSV
    if not csv_path.exists():
        print(f"Not found: {csv_path}")
        return None
    df = pd.read_csv(csv_path, sep='\t')
    print(f"Loaded {len(df)} rows, {df['OligoID'].nunique()} guides from {csv_path.name}")
    return df

trex2_df = load_profiles(TREX2_OUTPUT_DIR)
cas12a_df = load_profiles(CAS12A_OUTPUT_DIR)

## 2. Format Validation

In [ ]:
for name, output_dir in [('TREX2', TREX2_OUTPUT_DIR), ('Cas12a', CAS12A_OUTPUT_DIR)]:
    csv_path = Path(output_dir) / INDEL_PROFILES_CSV
    fasta_path = Path(output_dir) / REFERENCE_FASTA
    if not csv_path.exists():
        print(f"{name}: no data yet")
        continue
    result = validate_output_pair(csv_path, fasta_path)
    print(f"\n{name} Validation: {'PASS' if result['valid'] else 'FAIL'}")
    for err in result['csv']['errors'][:5]:
        print(f"  CSV: {err}")
    for err in result['fasta']['errors'][:5]:
        print(f"  FASTA: {err}")
    for err in result['cross_check_errors']:
        print(f"  Cross: {err}")

## 3. Indel String Parseability

In [ ]:
def check_parseability(df, name):
    """Check all indel strings are parseable."""
    if df is None:
        return
    valid = df['Identifier'].apply(validate_indel_string)
    n_invalid = (~valid).sum()
    print(f"{name}: {valid.sum()}/{len(df)} parseable ({n_invalid} invalid)")
    if n_invalid > 0:
        print("  Invalid examples:")
        for indel in df.loc[~valid, 'Identifier'].head(5):
            print(f"    {indel}")

check_parseability(trex2_df, 'TREX2')
check_parseability(cas12a_df, 'Cas12a')

## 4. Deletion Size Distributions

In [ ]:
def extract_del_sizes(df):
    """Extract deletion sizes from indel strings."""
    sizes = []
    counts = []
    for _, row in df.iterrows():
        try:
            itype, isize, details, muts = parse_indel_string(row['Identifier'])
            if itype == 'D':
                sizes.append(isize)
                counts.append(row['Count'])
        except Exception:
            pass
    return np.array(sizes), np.array(counts)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (df, name) in zip(axes, [(trex2_df, 'TREX2'), (cas12a_df, 'Cas12a')]):
    if df is None:
        ax.set_title(f'{name}: no data')
        continue
    sizes, counts = extract_del_sizes(df)
    if len(sizes) == 0:
        ax.set_title(f'{name}: no deletions')
        continue
    # Weighted histogram
    ax.hist(sizes, bins=range(0, 35), weights=counts, edgecolor='black', alpha=0.7)
    ax.set_xlabel('Deletion size (bp)')
    ax.set_ylabel('Total read count')
    ax.set_title(f'{name} Deletion Size Distribution\nMedian={np.median(np.repeat(sizes, counts)):.1f}bp')
    ax.axvline(np.median(np.repeat(sizes, counts)), color='red', linestyle='--', label='Median')
    ax.legend()

plt.tight_layout()
plt.savefig('../data/processed/deletion_size_distributions.png', dpi=150)
plt.show()

## 5. Deletion vs Insertion Fractions

In [ ]:
def get_indel_type_fractions(df):
    """Get fraction of reads that are deletions vs insertions."""
    if df is None:
        return None
    del_counts = 0
    ins_counts = 0
    other_counts = 0
    for _, row in df.iterrows():
        try:
            itype, _, _, _ = parse_indel_string(row['Identifier'])
            if itype == 'D':
                del_counts += row['Count']
            elif itype == 'I':
                ins_counts += row['Count']
            else:
                other_counts += row['Count']
        except Exception:
            other_counts += row['Count']
    total = del_counts + ins_counts + other_counts
    return {
        'Deletions': del_counts / total if total > 0 else 0,
        'Insertions': ins_counts / total if total > 0 else 0,
        'Other': other_counts / total if total > 0 else 0,
    }

for name, df in [('TREX2', trex2_df), ('Cas12a', cas12a_df)]:
    fracs = get_indel_type_fractions(df)
    if fracs:
        print(f"{name}: Deletions={fracs['Deletions']:.1%}, "
              f"Insertions={fracs['Insertions']:.1%}, Other={fracs['Other']:.1%}")

## 6. Reads-per-Guide Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (df, name) in zip(axes, [(trex2_df, 'TREX2'), (cas12a_df, 'Cas12a')]):
    if df is None:
        ax.set_title(f'{name}: no data')
        continue
    reads_per_guide = df.groupby('OligoID')['Count'].sum()
    ax.hist(reads_per_guide, bins=50, edgecolor='black', alpha=0.7)
    ax.set_xlabel('Total reads per guide')
    ax.set_ylabel('Number of guides')
    ax.set_title(f'{name}: Reads per Guide\nN={len(reads_per_guide)}, Median={reads_per_guide.median():.0f}')
    ax.axvline(500, color='red', linestyle='--', label='500-read threshold')
    ax.legend()

plt.tight_layout()
plt.savefig('../data/processed/reads_per_guide.png', dpi=150)
plt.show()

## 7. Feature Smoke Test

Run inDecay's `calculateFeatures` on 100 random indels from each dataset.

In [ ]:
# Add SelfTarget to path for feature extraction
sys.path.insert(0, '/Users/wergilius/Documents/SelfTarget/SelfTarget/indel_prediction')
sys.path.insert(0, '/Users/wergilius/Documents/SelfTarget/SelfTarget/selftarget_pyutils')

from Bio import SeqIO
from predictor.features import calculateFeatures

def smoke_test_features(df, fasta_path, name, n_samples=100):
    """Test feature extraction on random indels."""
    if df is None:
        print(f"{name}: no data")
        return
    fasta_path = Path(fasta_path)
    if not fasta_path.exists():
        print(f"{name}: no FASTA")
        return

    # Load reference sequences
    refs = {}
    for record in SeqIO.parse(str(fasta_path), 'fasta'):
        parts = str(record.description).split()
        oligo_id = parts[0]
        pam_loc = int(parts[1])
        pam_dir = parts[2]
        cut_site = pam_loc - 3 if pam_dir == 'FORWARD' else pam_loc + 3
        refs[oligo_id] = (str(record.seq), cut_site)

    # Sample random indels
    sample = df.sample(min(n_samples, len(df)), random_state=42)
    n_success = 0
    n_fail = 0

    for _, row in sample.iterrows():
        oligo_id = row['OligoID']
        indel = row['Identifier']
        if oligo_id not in refs:
            n_fail += 1
            continue

        seq, cut_site = refs[oligo_id]
        try:
            itype, isize, details, muts = parse_indel_string(indel)
            left = details.get('L', 0) + cut_site
            right = details.get('R', 0) + cut_site
            ins_seq = ''
            indel_details = (seq, cut_site, left, right, ins_seq)
            features, labels = calculateFeatures(indel_details)
            n_success += 1
        except Exception as e:
            n_fail += 1
            if n_fail <= 3:
                print(f"  Failed on {indel}: {e}")

    print(f"{name}: {n_success}/{n_success+n_fail} feature extractions succeeded")

smoke_test_features(trex2_df, TREX2_OUTPUT_DIR / REFERENCE_FASTA, 'TREX2')
smoke_test_features(cas12a_df, CAS12A_OUTPUT_DIR / REFERENCE_FASTA, 'Cas12a')

## 8. Summary Statistics

In [ ]:
def summary_stats(df, name):
    if df is None:
        print(f"{name}: no data")
        return
    print(f"\n=== {name} ===")
    print(f"Total rows: {len(df)}")
    print(f"Unique guides: {df['OligoID'].nunique()}")
    print(f"Unique indels: {df['Identifier'].nunique()}")
    reads_per_guide = df.groupby('OligoID')['Count'].sum()
    print(f"Reads per guide: median={reads_per_guide.median():.0f}, "
          f"mean={reads_per_guide.mean():.0f}, "
          f"min={reads_per_guide.min()}, max={reads_per_guide.max()}")
    indels_per_guide = df.groupby('OligoID')['Identifier'].nunique()
    print(f"Unique indels per guide: median={indels_per_guide.median():.0f}, "
          f"mean={indels_per_guide.mean():.0f}")

summary_stats(trex2_df, 'TREX2')
summary_stats(cas12a_df, 'Cas12a')